1. What RAG is, and why

An LLM doesn’t know your private documents, has a knowledge cutoff, and can invent facts. RAG fixes this in three steps:

|Step|What happens|In our project
|---|---|---|
|retrieve|Find the chunks most relevant to the question|Your EvidenceIndex.search (Day 22)
|Augment|Put those chunks into the prompt|Next step
|Generate|The LLM answers using only those chunks|Gemini, with citations

*Why not fine-tuning? Fine-tuning teaches style and behaviour, not facts. RAG updates by changing documents, shows its sources, and lets you check the answer against them*

2. Two phases

|Offline (once per document)|Online (every question)
|---|---|
|parse → chunk → embed → index|embed query → search → build prompt → LLM → validate

3. The prompt is a contract

A production RAG prompt does these things:

- Says “answer only from the context below”.
- Gives a way out: “if the context doesn’t contain the answer, say NOT_FOUND”. Without that, the model guesses.
- Labels each chunk with its id, like [ev_013] ..., and requires the answer to cite ids.
- Asks for structured JSON (Pydantic, from Day 21) with temperature=0.
- Is followed by code that checks every cited id was actually retrieved, because models can invent citations.

4. Two separate things to evaluate

|Part|Question|Metrics
|---|---|---|
|Retrieval|Did the right chunk get into the context?|Hit@k, Recall@k, MRR (you built these)
|Generation|Is the answer faithful to the context, and are the citations real?|Faithfulness, citation accuracy, correctness of the verdict

Importan: *Debug in this order: if retrieval fails, generation can’t be right. Check retrieval first.*

5. How it becomes the tender system

For each requirement: retrieve evidence → the LLM returns a verdict (supported, unsupported, partial) with a reason and cited ids → code verifies the ids. This is the Day 21 rule applied again: the model decides, and the code checks.

Exercises (short answers are fine)

- E1. Name the three RAG steps in order. Which of them does your Day 22 code already do?
   * retrieve, augment and generate. We already did the retrieve part (EvidenceIndex.search)

- E2. Classify each step as offline or online: chunk the documents, embed the documents, embed the query, FAISS search, build the prompt, LLM generate.
   * offline: chunk -> embed -> index
   * online: embed query -> FAISS search -> build prompt -> LLM generate 

- E3. Write 4 rules for the system instruction of the tender judge prompt. Hint: use points 3 above.
   * Says the answer only from the context.
   * If the context does not contain the answer say NOT FOUND.
   * Labels each chunk with its id and requires the answer to cite ids
   * Asked for structured JSON

- E4. For R06 (“professional indemnity insurance of not less than RM 10 million”), the retrieved chunks are ev_013 (RM 5 million PI) and ev_014 (RM 2 million public liability). Write the verdict, the cited id and a one-sentence reason the model should output.
   * verdict: unsupported, cited_id: ev_013, 
   * output: The bidder does not meet the requirement because their professional indemnity insurance is only RM 5 million, which falls short of the required minimum of RM 10 million. 

- E5. For each symptom, say whether you check retrieval or generation first, and why:

   * (a) The answer cites ev_099, which doesn’t exist.
      - generation because generation, because the model invented an id that retrieval never returned. Fix it with a check that every cited id  was in the retrieved set. 
   * (b) The right chunk is not in the top 5.
      - retrieval, because the chunk never reached the context, so generation cannot succeed. Check the query wording and k.
   * (c) The right chunk is in the context but the model says NOT_FOUND.
      - generation, check the LLM prompt or maybe the model ignored the context.
 
- E6. Which of these measure retrieval, and which measure generation: Hit@k, faithfulness, MRR, citation accuracy?
   * measure retrieval: Hit@k, MRR
   * measure generation: citation accuracy, faithfulness

- Load: turn a file (PDF) into text plus metadata (file name, page number). Without that metadata you can’t cite, and citing is the whole point of our judge.

- Chunk: cut the text into pieces small enough to retrieve precisely. Too big and one chunk holds many topics, so the match gets blurry. Too small and a fact is split across two chunks, so neither chunk answers the question. Overlap (repeating a little text at the boundary) reduces that splitting.

- Your two questions before any code (short answers, your own words):

- E1. You have a 2-page PDF where “Professional indemnity insurance: RM 5 million” ends at the very end of a chunk and “valid until 2027” starts the next chunk. Which problem is this, and what two things could you change to reduce it?
   * The cut landed in the middle of a fact, so the fact is split across two chunks. It’s a boundary-splitting problem, usually caused by chunks that are too small or cut at arbitrary places.. Split the chunk to overlap so it can repeat a littile text at the boundary so that it can reduces that splitting. Split at natural boundaries (paragraph, then sentence) instead of at a fixed character count. A bigger chunk size also helps.

- E2. After chunking, ev_013 no longer exists, because the chunks are new. Your 25 retrieval-eval questions label relevant_ids by chunk id. What breaks, and what could you label with instead? (Think about this one. It’s the real production problem.)
   * The labels still point to ev_013, which no longer exists. So Hit@k and Recall@k fall to 0 even when retrieval is right. The metric breaks. We can label as parent id instead. Label with the answer text (for example “RM 5 million”). A retrieved chunk counts as relevant if its text contains that phrase. This is the most robust option.